# 12_react_rag_agent

12_react_rag_agent.py — ReAct 패턴의 RAG 에이전트 (비교군)

10_build_and_run.py 의 *명시적 그래프* 와 대비.
ReAct = 검색을 "도구" 로 두고 LLM 이 생각 → 검색 → 관찰 → 다시 생각 을 자율적으로 반복.
유연하지만 예측 가능성이 낮고 비용이 들쭉날쭉한 단점.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '12_react_rag_agent.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
12_react_rag_agent.py — ReAct 패턴의 RAG 에이전트 (비교군)

10_build_and_run.py 의 *명시적 그래프* 와 대비.
ReAct = 검색을 "도구" 로 두고 LLM 이 생각 → 검색 → 관찰 → 다시 생각 을 자율적으로 반복.
유연하지만 예측 가능성이 낮고 비용이 들쭉날쭉한 단점.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from _common import get_llm, get_vectorstore, SAMPLE_DOCS, web_search_fn, banner, llm_unavailable


_retriever = get_vectorstore(SAMPLE_DOCS).as_retriever(search_kwargs={"k": 4})


@tool
def search_knowledge_base(query: str) -> str:
    """사내 지식베이스(벡터 DB)에서 관련 문서를 검색한다.
    AI 에이전트 메모리·도구 사용·Self-RAG·CRAG 등 RAG 관련 질문에 사용."""
    docs = _retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)


@tool
def search_web(query: str) -> str:
    """최신 뉴스·시세·실시간 정보가 필요한 경우 외부 웹을 검색한다."""
    return web_search_fn(query, k=3)


def main() -> None:
    llm = get_llm()
    if llm is None:
        llm_unavailable()
        return

    agent = create_react_agent(
        model=llm,
        tools=[search_knowledge_base, search_web],
    )

    banner("ReAct Agent — 검색을 '도구' 로 두고 자율적으로 호출")
    result = agent.invoke({
        "messages": [("user", "에이전트 메모리 종류를 정리해줘. 한국어로 간결히.")],
    })

    print("\n💬 메시지 trace (간략):")
    for msg in result["messages"]:
        kind = type(msg).__name__
        content_preview = (msg.content or "")[:120].replace("\n", " ")
        tool_calls = getattr(msg, "tool_calls", None) or []
        if tool_calls:
            for tc in tool_calls:
                print(f"  [{kind}] tool_call → {tc['name']}({tc['args']})")
        else:
            print(f"  [{kind}] {content_preview}")

    print("\n📝 최종 답변")
    print("-" * 70)
    print(result["messages"][-1].content)


if __name__ == "__main__":
    main()

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7488.35it/s]

C:\Users\user\AppData\Local\Temp\ipykernel_46192\1204843179.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



📌 ReAct Agent — 검색을 '도구' 로 두고 자율적으로 호출



💬 메시지 trace (간략):
  [HumanMessage] 에이전트 메모리 종류를 정리해줘. 한국어로 간결히.
  [AIMessage] tool_call → search_knowledge_base({'query': 'AI 에이전트 메모리 종류'})
  [ToolMessage] LLM 에이전트의 메모리는 크게 세 가지로 나뉜다. 단기 메모리(short-term)는 현재 대화의 컨텍스트 윈도우이고, 장기 메모리(long-term)는 벡터DB나 외부 저장소에 보관해 필요할 때 검색해 가져오며,
  [AIMessage] ## 🤖 AI 에이전트 메모리 종류  | 구분 | 설명 | 저장 방식 | |------|------|-----------| | **감각 메모리 (Sensory Memory)** | 입력 직후 아주 짧은 순간의 원시 

📝 최종 답변
----------------------------------------------------------------------
## 🤖 AI 에이전트 메모리 종류

| 구분 | 설명 | 저장 방식 |
|------|------|-----------|
| **감각 메모리 (Sensory Memory)** | 입력 직후 아주 짧은 순간의 원시 데이터를 보관 | 즉시 소멸, 처리 후 단기로 이동 |
| **단기 메모리 (Short-Term Memory)** | 현재 대화/세션의 컨텍스트 (프롬프트 윈도우) | 컨텍스트 윈도우 내 유지 |
| **장기 메모리 (Long-Term Memory)** | 과거 경험·지식을 영구 저장, 필요 시 검색 | 벡터 DB, 외부 저장소 |

---

### 🔍 추가 설명

- **단기 메모리** = LLM의 컨텍스트 윈도우(token limit)로 제한됨
- **장기 메모리** = 벡터 임베딩으로 저장 → RAG(Retrieval-Augmented Generation) 방식으로 검색해 활용
- **감각 메모리** = 사람의 감각 기관처럼 입력을 받자마자 매우 짧게 유지되며, 중요한 정보만 